# In-hospital mortality with GPT2 and Mamba (sequence classifiers)

Reconstruction of `transformers/models.py` (`CustomGPT`, `CustomMamba`) + `transformers/trainer.py` for the MIMIC-IV
mortality task built by `01_prepare_data.ipynb`.

**Architecture (same for both backbones)**
1. Input = `[gender, race, age] + bucketed events` (token ids), plus the event time in hours since ICU admission.
2. `combine_layer(concat(token_embedding, time_layer(time)))` → `inputs_embeds` of the pretrained backbone.
3. Hidden states are pooled over the non-padding positions (mean by default) → MLP head → 1 logit.
4. Loss: `BCEWithLogitsLoss` with `pos_weight = #neg / #pos`. The best epoch is picked on validation AUROC and reported on test.

| | GPT2 (`gpt2`, 124M) | Mamba (`state-spaces/mamba-130m-hf`) |
|---|---|---|
| fine-tuning | last 2 transformer blocks + final LayerNorm | last 2 Mamba blocks + final norm |
| context | 1024 tokens | 2048 tokens (needs `mamba-ssm` + `causal-conv1d` CUDA kernels to be practical) |

In both cases the token embeddings (new event vocabulary), the time / combine layers and the head are trained.
Differences from the original scripts:
* The embeddings are trainable here. The original froze them while also resizing them to a brand-new vocabulary.
* Mamba fine-tunes its last blocks (as in `pretrain/models.py`) instead of using LoRA. With the CUDA kernels,
  HF's training path passes the `x_proj` / `dt_proj` / `out_proj` weights straight into `mamba_inner_fn`, so LoRA
  adapters on those modules are silently skipped and never train.
* Model selection uses the validation split instead of the test split.
* The demographic tokens are actually fed to the model. The original `trainer.py` crashed on the 4-tuple batch.

In [ ]:
import sys
from collections import OrderedDict
from pathlib import Path

import torch
import torch.nn as nn

sys.path.insert(0, str(Path.cwd()))
from mortality_common import (
    CHECKPOINT_DIR, set_seed, get_device, load_tokenizer, make_loaders, encode_batch,
    pool_hidden_states, count_parameters, train_binary_classifier, save_results,
)

In [ ]:
# ---------------- configuration ----------------
MODEL_NAME = 'gpt2'          # 'gpt2' or 'mamba'
SEED = 1234
GPU = 0
POOLING = 'mean'             # 'mean' | 'max' | 'last' | 'first'
N_EPOCHS = 10
PATIENCE = 3                 # early stopping on validation AUROC
LEARNING_RATE = 5e-5
GRAD_ACCUM = 2
INIT_FROM_CHECKPOINT = None  # optional: LM checkpoint from 03_gpt2_ar.ipynb stage 1 (gpt2 only), e.g. CHECKPOINT_DIR / 'gpt2_lm_pretrain.pth'

if MODEL_NAME == 'gpt2':
    PRETRAINED = 'gpt2'
    CONTEXT_LENGTH = 1024    # GPT2 max positions (demographic tokens included)
    BATCH_SIZE = 8
else:
    PRETRAINED = 'state-spaces/mamba-130m-hf'
    CONTEXT_LENGTH = 2048
    BATCH_SIZE = 8

set_seed(SEED)
device = get_device(GPU)
tokenizer = load_tokenizer()
N_DEMO = 3
loaders = make_loaders(BATCH_SIZE, max_events=CONTEXT_LENGTH - N_DEMO, eval_batch_size=2 * BATCH_SIZE)
print('device:', device, '| vocab:', len(tokenizer))

## Models

In [ ]:
from transformers import GPT2Model, MambaModel


class GPT2MortalityClassifier(nn.Module):
    def __init__(self, vocab_size, pretrained='gpt2', n_trainable_blocks=2, pooling='mean', dropout=0.1):
        super().__init__()
        self.backbone = GPT2Model.from_pretrained(pretrained)
        self.backbone.resize_token_embeddings(vocab_size)
        hidden_size = self.backbone.config.n_embd

        if n_trainable_blocks is not None:  # None = full fine-tuning
            for p in self.backbone.parameters():
                p.requires_grad = False
            for block in self.backbone.h[-n_trainable_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
            for module in [self.backbone.ln_f, self.backbone.wte]:
                for p in module.parameters():
                    p.requires_grad = True

        self.time_layer = nn.Linear(1, hidden_size)
        self.combine_layer = nn.Linear(hidden_size * 2, hidden_size)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1),
        )
        self.pooling = pooling

    def forward(self, input_ids, times, attention_mask):
        token_embeddings = self.backbone.wte(input_ids)
        time_embeddings = self.time_layer(times.unsqueeze(-1))
        embs = self.combine_layer(torch.cat([token_embeddings, time_embeddings], dim=-1))
        hidden = self.backbone(inputs_embeds=embs, attention_mask=attention_mask).last_hidden_state
        return self.head(pool_hidden_states(self.pooling, hidden, attention_mask)).squeeze(-1)


class MambaMortalityClassifier(nn.Module):
    def __init__(self, vocab_size, pretrained='state-spaces/mamba-130m-hf', n_trainable_blocks=2, pooling='mean', dropout=0.1):
        super().__init__()
        self.backbone = MambaModel.from_pretrained(pretrained)
        self.backbone.resize_token_embeddings(vocab_size)
        hidden_size = self.backbone.config.hidden_size

        if n_trainable_blocks is not None:  # None = full fine-tuning
            for p in self.backbone.parameters():
                p.requires_grad = False
            for block in self.backbone.layers[-n_trainable_blocks:]:
                for p in block.parameters():
                    p.requires_grad = True
            for module in [self.backbone.norm_f, self.backbone.embeddings]:
                for p in module.parameters():
                    p.requires_grad = True

        self.time_layer = nn.Linear(1, hidden_size)
        self.combine_layer = nn.Linear(hidden_size * 2, hidden_size)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, 1),
        )
        self.pooling = pooling

    def forward(self, input_ids, times, attention_mask):
        token_embeddings = self.backbone.embeddings(input_ids)
        time_embeddings = self.time_layer(times.unsqueeze(-1))
        embs = self.combine_layer(torch.cat([token_embeddings, time_embeddings], dim=-1))
        # Mamba is causal and the batch is right-padded, so padding never affects real positions;
        # the mask is only needed for pooling.
        hidden = self.backbone(inputs_embeds=embs).last_hidden_state
        return self.head(pool_hidden_states(self.pooling, hidden, attention_mask)).squeeze(-1)

In [ ]:
if MODEL_NAME == 'gpt2':
    model = GPT2MortalityClassifier(len(tokenizer), PRETRAINED, pooling=POOLING)
    if INIT_FROM_CHECKPOINT is not None:
        # GPT2EventLM (03_gpt2_ar) keys look like 'model.transformer.h.0...' -> 'backbone.h.0...'
        lm_state = torch.load(INIT_FROM_CHECKPOINT, map_location='cpu', weights_only=True)
        state = OrderedDict((k.replace('model.transformer.', 'backbone.'), v) for k, v in lm_state.items()
                            if k.startswith('model.transformer.'))
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'loaded {len(state)} tensors from {INIT_FROM_CHECKPOINT} (missing {len(missing)}, unexpected {len(unexpected)})')
else:
    import importlib.util
    if not all(importlib.util.find_spec(m) for m in ['mamba_ssm', 'causal_conv1d']):
        print('WARNING: mamba-ssm / causal-conv1d not installed -> HF may fall back to the slow, memory-hungry '
              'sequential scan. Install them (pip install mamba-ssm causal-conv1d) or lower CONTEXT_LENGTH / BATCH_SIZE.')
    model = MambaMortalityClassifier(len(tokenizer), PRETRAINED, pooling=POOLING)

model = model.to(device)
count_parameters(model)


def predict_fn(model, batch, train):
    input_ids, times, attention_mask = encode_batch(batch, tokenizer.pad_token_id, device)
    return model(input_ids, times, attention_mask)

## Train / evaluate

In [ ]:
run_name = f'{MODEL_NAME}_seed{SEED}'
history, test_metrics, test_pred = train_binary_classifier(
    model, predict_fn, loaders, device, CHECKPOINT_DIR / f'{run_name}.pth',
    learning_rate=LEARNING_RATE, n_epochs=N_EPOCHS, patience=PATIENCE, grad_accum=GRAD_ACCUM,
)

In [ ]:
config = dict(model=MODEL_NAME, pretrained=PRETRAINED, seed=SEED, context_length=CONTEXT_LENGTH, batch_size=BATCH_SIZE,
              grad_accum=GRAD_ACCUM, learning_rate=LEARNING_RATE, pooling=POOLING, init_from=str(INIT_FROM_CHECKPOINT))
save_results(run_name, test_metrics, test_pred, history, config)
history